In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import shap

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import DIVERGING_CMAP, configure_mpl
from ising import Ising, SymmetricIsing

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")

schema = schema.post_index()

Load data

When calculating the effect of intervention, we should consider people's prior state. For instance, _given that_ individual $i$ previously didn't believe in climate change, what is the effect of intervening?

In [ ]:
covariates = False

cov_flag = "yes_use_covariates" if covariates else "no_use_covariates"

no_int_asym_results = np.load(DATA_PATH / f"ising_00_{cov_flag}.npz")
int_asym_results = np.load(DATA_PATH / f"ising_05_{cov_flag}.npz")
no_int_sym_results = np.load(DATA_PATH / f"sym_ising_00_{cov_flag}.npz")
int_sym_results = np.load(DATA_PATH / f"sym_ising_05_{cov_flag}.npz")

Y0 = no_int_asym_results["Y0"]
Y = no_int_asym_results["Y"]
if covariates:
    X = no_int_asym_results["X"]
    K = X.shape[-1]
else:
    X = None
    K = 0


asym_params = no_int_asym_results["params"]
sym_params = no_int_sym_results["params"]

asym_interaction_effects = np.asarray(
    [Ising.unpack_params(params, k=K)[1] for params in asym_params]
)
sym_interaction_effects = [
    SymmetricIsing.unpack_params(params, k=K)[1] for params in sym_params
]
sym_interaction_effects = np.asarray([J + J.T for J in sym_interaction_effects])

t = 4
no_int_asym_measurements = no_int_asym_results["measurements"][:, :, t]
int_asym_measurements = int_asym_results["measurements"][:, :, t]
no_int_sym_measurements = no_int_sym_results["measurements"][:, :, t]
int_sym_measurements = int_sym_results["measurements"][:, :, t]

n = Y0.shape[-1]
for i in range(n):
    no_int_asym_measurements[..., i, i] = 0
    int_asym_measurements[..., i, i] = 0
    no_int_sym_measurements[..., i, i] = 0
    int_sym_measurements[..., i, i] = 0

int_effect_asym = int_asym_measurements - no_int_asym_measurements
int_effect_sym = int_sym_measurements - no_int_sym_measurements
asym_effect = int_effect_asym - int_effect_sym

cols = schema.get_short_names(kind="measurement")

### Shapley values

For each intervention (X --> Y), calculate the average intervention effect (intervention outcome - null outcome). Calculate shapley values, taking the initial state of each spin as the features.

In [ ]:
cols

Considering 'CC Worry', targeting 'CC Anthropogenic'.

In [ ]:
effect = int_effect_asym[:, :, 1, 2].mean(axis=1)
initial_state = Y0[:, -1]

h, j = Ising.unpack_params(asym_params[0], k=0)
adj = np.full_like(j, fill_value=True, dtype=np.bool)
adj[7] = False
adj[7, 7] = True
model = Ising(field=h, coupling=j, connectivity=adj, rng=RANDOM_SEED)
imodel = model.intervene(
    spins=np.array([2]), field_offset=np.array([1.0]), seed=RANDOM_SEED
)

seed = RANDOM_SEED + 1


def s_anthro(_model, y):
    return y[1]


def s_policy(_model, y):
    return y[7]


def intervention_effect(x):
    Y0 = x.values
    # print(np.unique(Y0))
    # print(Y.shape)
    global seed
    imodel.reset(seed)
    model.reset(seed)
    seed += 1
    int_outcome = imodel.measure(s_policy, y0=Y0, t=5, repeats=3, warmup_steps=0).y[
        ..., -1
    ]
    null_outcome = model.measure(s_policy, y0=Y0, t=5, repeats=3, warmup_steps=0).y[
        ..., -1
    ]
    int_effect = (int_outcome - null_outcome).mean(axis=-1)
    return int_effect
    # print(x)
    # return effect


# fig, ax = shap.partial_dependence_plot(
#     "Politics",
#     intervention_effect,
#     df.head(10),
#     model_expected_value=True,
#     feature_expected_value=True,
#     show=False,
#     ice=False,
# )

In [ ]:
df = pl.DataFrame(Y[0, :, -1], schema=cols).to_pandas()
explainer = shap.Explainer(
    intervention_effect, df.sample(n=50), algorithm="permutation"
)
shap_values = explainer(df.sample(400))

In [ ]:
shap_values.shape

In [ ]:
plot_df = pl.DataFrame(
    {
        "Spin": [c for _ in range(shap_values.shape[0]) for c in cols],
        "Shapley value": shap_values.values.flatten(),
        "State": shap_values.data.flatten(),
    }
)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
# sns.stripplot(shap_values.values, orient="h", s=3, ax=ax)
sns.stripplot(
    plot_df,
    y="Spin",
    x="Shapley value",
    hue="State",
    orient="h",
    s=3,
    alpha=0.75,
    ax=ax,
)

ax.set_yticks(np.arange(8), cols);
# ax.set_xlim(-0.035, 0.035)

In [ ]:
fig.savefig("../meeting_6_11/shapley_cc_worry_on_policy.pdf", bbox_inches="tight")

## Determining most effective intervention for each target node

For each target spin, record the proportion of times each possible intervention spin yields the highest mean outcome across individuals. 

In [ ]:
def top_strategy_props(measurements):
    avg_state = measurements.mean(axis=0)
    intervention_spin_ranks = avg_state.argsort(axis=-1)
    top_strategy = intervention_spin_ranks[..., -1]
    r = intervention_spin_ranks.shape[0]
    n = intervention_spin_ranks.shape[1]
    prop_top_strategy = np.zeros((n, n), dtype=np.float64)
    for target in range(n):
        for intervention in range(n):
            prop_top_strategy[target, intervention] = (
                top_strategy[..., target] == intervention
            ).sum() / r
    return prop_top_strategy

In [ ]:
top_strategy_props(int_asym_measurements)

In [ ]:
top_strategy_props(int_sym_measurements)

Consider the effect of intervening on politics to affect belief in CC

In [ ]:
mean_effect_by_ind_asym = int_effect_asym.mean(axis=1)
np.percentile(mean_effect_by_ind_asym[..., 7, 2], q=(10, 90, 95))

In [ ]:
np.argwhere(mean_effect_by_ind_asym[..., 7, 2] > 0.16).shape

## Who is 'CC Worry -> Climate policy' effective for?

In [ ]:
state_counts = dict()
top_effect_states = (
    Y[:, np.argwhere(mean_effect_by_ind_asym[..., 7, 2] > 0.16)[:, 0], -1]
    .ravel()
    .reshape((-1, 8))
)

for state in top_effect_states:
    state_tuple = tuple([int(s) for s in state])
    state_counts[state_tuple] = state_counts.get(state_tuple, 0) + 1

In [ ]:
states = np.empty((len(state_counts), 8), dtype=np.int64)
counts = np.empty(len(state_counts), dtype=np.int64)
for i, (state, count) in enumerate(state_counts.items()):
    states[i] = np.asarray(state)
    counts[i] = count

sort_idx = np.argsort(counts)[::-1]
states = states[sort_idx]
counts = counts[sort_idx]

In [ ]:
states[:3], counts[:3]

### Modal individual: Democrat who believes in climate change, but does not think it is too serious, and doesn't support climate policy.
- Believes in climate change
- Believes climate change is human-caused
- Not worried about climate change
- Does not think others are worried about climate change
- Is not worried about extreme weather
- Is democrat-aligned
- Does not think climate change impacts are high
- Does not support climate policy

### Second-most-common: Republican who believes in climate change and is worried about it, who doesn't believe others are, and doesn't think current impacts are high, but nonetheless supports policy
- Believes in climate change
- Believes climate change is human-caused
- Worried about climate change
- Does not think others are worried about climate change
- Is not worried about extreme weather
- Is republican-aligned
- Does not think climate change impacts are high
- Supports climate policy

### Third-most-common: Republican who believes in climate change and is worried about it, who doesn't believe others are, but who believes the current impacts are high and supports policy
- Believes in climate change
- Believes climate change is human-caused
- Worried about climate change
- Does not think others are worried about climate change
- Is not worried about extreme weather
- Is republican-aligned
- Thinks climate change impacts are high
- Supports climate policy

## Who is 'CC Worry -> Climate policy' _not_ effective for?

In [ ]:
state_counts = dict()
low_effect_states = (
    Y[:, np.argwhere(mean_effect_by_ind_asym[..., 7, 2] <= 0.2)[:, 0], -1]
    .ravel()
    .reshape((-1, 8))
)

for state in low_effect_states:
    state_tuple = tuple([int(s) for s in state])
    state_counts[state_tuple] = state_counts.get(state_tuple, 0) + 1

In [ ]:
states = np.empty((len(state_counts), 8), dtype=np.int64)
counts = np.empty(len(state_counts), dtype=np.int64)
for i, (state, count) in enumerate(state_counts.items()):
    states[i] = np.asarray(state)
    counts[i] = count

sort_idx = np.argsort(counts)[::-1]
states = states[sort_idx]
counts = counts[sort_idx]

In [ ]:
states[:6], counts[:6]

### Modal individual: Democrat who believes in climate change, is worried about it but doesn't think other are, thinks impacts are high, and supports climate policy
- Believes in climate change
- Believes climate change is human-caused
- Worried about climate change
- Does not think others are worried about climate change
- Is worried about extreme weather
- Is democrat-aligned
- Thinks climate change impacts are high
- Supports climate policy

### Second-most-common: Democrat who believes in climate change, is worried about it and thinks others are as well, thinks impacts are high, and supports climate policy
- Believes in climate change
- Believes climate change is human-caused
- Worried about climate change
- Thinks others are worried about climate change
- Is worried about extreme weather
- Is democrat-aligned
- Thinks climate change impacts are high
- Supports climate policy

## Fourth most-common: Republican who doesn't believe in climate change, isn't worried about it or extreme weather, and doesn't support climate policy
- Does not believe in climate change
- Does not believe climate change is human-caused
- Is not worried about climate change
- Does not think others are worried about climate change
- Is not worried about extreme weather
- Is republican-aligned
- Thinks climate change impacts are low
- Does not supports climate policy

## Sixth most-common: Republican who believes in climate change and thinks it is human-caused, but isn't worried, doesn't think the impacts are high, and doesn't support climate policy
- Believes in climate change
- Believes climate change is human-caused
- Is not worried about climate change
- Does not think others are worried about climate change
- Is not worried about extreme weather
- Is republican-aligned
- Thinks climate change impacts are low
- Does not supports climate policy

In [ ]:
mean_effect_by_ind_asym = int_effect_asym.mean(axis=1)
max_effect_size = np.percentile(abs(mean_effect_by_ind_asym), q=99.9, axis=(0, 1)).max()
candidates = np.linspace(0, 1, 21)
xmax = candidates[np.argmax(candidates >= max_effect_size)]
for i, col in enumerate(cols):
    fig, axes = plt.subplots(
        nrows=3, ncols=3, constrained_layout=True, sharey=True, sharex=True
    )
    for j, (other_col, ax) in enumerate(zip(cols, axes.flatten(), strict=True)):
        if i == j:
            continue
        elif j == 7:
            # NOTE: This is hard-coded at the moment; 7 <-- policies, which we have
            #   made a sink.
            continue
        sns.histplot(
            mean_effect_by_ind_asym[..., i, j], stat="probability", ax=ax, binwidth=0.02
        )
        ax.set_title(f"{other_col}")
        ax.set_xlim(-xmax, xmax)
    fig.suptitle(col)
    plt.show()

In [ ]:
mean_effect_by_ind_asym = int_effect_asym.mean(axis=1)
effect = np.delete(mean_effect_by_ind_asym, i, axis=-1)
for i, col in enumerate(cols):
    fig, axes = plt.subplots(
        nrows=3, ncols=3, constrained_layout=True, sharey=True, sharex=True
    )
    for j, (other_col, ax) in enumerate(zip(cols, axes.flatten(), strict=True)):
        if i == j:
            continue
        elif j == 7:
            # NOTE: This is hard-coded at the moment; 7 <-- policies, which we
            #   have made a sink.
            continue
        sns.histplot(
            mean_effect_by_ind_asym[..., i, j], stat="probability", ax=ax, binwidth=0.02
        )
        ax.set_title(f"{other_col}")
        # ax.set_xlim(-0.75, 0.75)
    fig.suptitle(col)
    plt.show()

## Who is it effective for?

Look at effect of intervening on 'weather worry' to affect 'climate impacts' target. Which individuals is it effective for? Which is it not?

## Illustrate variation in effect of intervention on different target nodes

Look at effects of intervening on 'Belief that CC is anthropogenic'

### 1. How does measured effect vary at different target nodes?

In [ ]:
def bootstrap_mean_effects(measurements, rng, n_boot=100, ci=95):
    # measurements: shape (n_individuals, repeats, n_interventions)
    M, R, N = measurements.shape
    boot_means = np.empty((n_boot, N))

    for b in range(n_boot):
        sample_rows = rng.integers(0, M, M)
        samples = measurements[sample_rows]

        # Final estimate is the average rank per intervention spin
        boot_means[b] = samples.mean(axis=(0, 1))

    mean = boot_means.mean(axis=0)
    ci = 1.97 * boot_means.std(axis=0, ddof=1) / np.sqrt(n_boot)
    lower = mean - ci
    upper = mean + ci

    return mean, lower, upper


i = 1

asym_mean, asym_lo, asym_hi = bootstrap_mean_effects(int_effect_asym[..., i], rng)
sym_mean, sym_lo, sym_hi = bootstrap_mean_effects(int_effect_sym[..., i], rng)

asym_mean = np.delete(asym_mean, i)
asym_lo = np.delete(asym_lo, i)
asym_hi = np.delete(asym_hi, i)
sym_mean = np.delete(sym_mean, i)
sym_lo = np.delete(sym_lo, i)
sym_hi = np.delete(sym_hi, i)

plot_df = pl.DataFrame(
    {
        "Model": ["Asymmetric"] * (n - 1) + ["Symmetric"] * (n - 1),
        "Target": np.concat((np.delete(cols, i), np.delete(cols, i))),
        "Effect": np.concat((asym_mean, sym_mean)),
        "CI low": np.concatenate([asym_lo, sym_lo]),
        "CI high": np.concatenate([asym_hi, sym_hi]),
    }
)

fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
sns.barplot(plot_df.to_pandas(), x="Target", y="Effect", hue="Model", ax=ax)

ax.set_xticks(
    np.arange(n - 1), np.delete(cols, i), rotation=30, horizontalalignment="right"
)

x = np.arange(len(cols) - 1)
width = 0.4

for j, model in enumerate(["Asymmetric", "Symmetric"]):
    subset = plot_df.filter(Model=model)
    offset = -width / 2 if j == 0 else width / 2

    ax.errorbar(
        x + offset,
        subset["Effect"],
        yerr=[
            subset["Effect"] - subset["CI low"],
            subset["CI high"] - subset["Effect"],
        ],
        fmt="none",
        capsize=4,
        color="black",
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ax.set_ylim(-0.025, 0.1)

leg = ax.get_legend()
handles = leg.legend_handles
labels = [t.get_text() for t in leg.texts]

# ax.legend(handles, labels)
leg.remove()

ax.legend(
    handles,
    labels,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    # fontsize=8,
    # handlelength=1,
    # columnspacing=0.5,
    # labelspacing=0.2,
    frameon=False,
)

ax.set_title(f"Effects of intervening on '{cols[i]}'", pad=30)

### 2. How does intervention effect compare to interaction weight?

Plot intervention effect against interaction weight. Does it form a straight line?

In [ ]:
i = 2

asym_mean_interaction_effect = np.delete(asym_interaction_effects[:, i].mean(axis=0), i)
sym_mean_interaction_effect = np.delete(sym_interaction_effects[:, i].mean(axis=0), i)

asym_mean, *_ = bootstrap_mean_effects(int_effect_asym[..., i], rng)
sym_mean, *_ = bootstrap_mean_effects(int_effect_sym[..., i], rng)

asym_mean = np.delete(asym_mean, i)
sym_mean = np.delete(sym_mean, i)

plot_df = pl.DataFrame(
    {
        "Model": ["Asymmetric"] * (n - 1) + ["Symmetric"] * (n - 1),
        "Intervention effect": np.concat((asym_mean, sym_mean)),
        "Interaction effect": np.concat(
            (asym_mean_interaction_effect, sym_mean_interaction_effect)
        ),
    }
)

sns.relplot(plot_df, x="Interaction effect", y="Intervention effect", hue="Model")

### 3. How do the effects change over time?

In [ ]:
i = 2

no_int_asym_measurements_ts = no_int_asym_results["measurements"][..., i]
int_asym_measurements_ts = int_asym_results["measurements"][..., i]
no_int_sym_measurements_ts = no_int_sym_results["measurements"][..., i]
int_sym_measurements_ts = int_sym_results["measurements"][..., i]


no_int_asym_measurements_ts[..., i] = 0
int_asym_measurements_ts[..., i] = 0
no_int_sym_measurements_ts[..., i] = 0
int_sym_measurements_ts[..., i] = 0

int_effect_asym_ts = int_asym_measurements_ts - no_int_asym_measurements_ts
int_effect_sym_ts = int_sym_measurements_ts - no_int_sym_measurements_ts

asym_effect_ts = int_effect_asym_ts - int_effect_sym_ts

mean_int_effect_asym_ts = (int_effect_asym_ts).mean(axis=(0, 1))
asym_lo, asym_hi = np.percentile((int_effect_asym_ts).mean(axis=0), q=(5, 95), axis=0)
mean_int_effect_sym_ts = (int_effect_sym_ts).mean(axis=(0, 1))
sym_lo, sym_hi = np.percentile((int_effect_sym_ts).mean(axis=0), q=(5, 95), axis=0)

fig, axes = plt.subplots(
    ncols=2, figsize=(5.5, 2), constrained_layout=True, sharey=True
)

select_idxes = [3, 4, 7]
t = mean_int_effect_asym_ts.shape[0]
for col_idx in select_idxes:
    axes[0].plot(
        np.arange(t), mean_int_effect_asym_ts[:, col_idx], label=f"{cols[col_idx]}"
    )
    axes[0].fill_between(
        np.arange(t), asym_lo[:, col_idx], asym_hi[:, col_idx], alpha=0.3
    )

for col_idx in select_idxes:
    axes[1].plot(
        np.arange(t), mean_int_effect_sym_ts[:, col_idx], label=f"{cols[col_idx]}"
    )
    axes[1].fill_between(
        np.arange(t), sym_lo[:, col_idx], sym_hi[:, col_idx], alpha=0.3
    )

for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Time")
    ax.set_xlim(0, 9)
axes[0].set_ylabel("Effect of intervention")

axes[0].set_title("Aymmetric", fontsize=11)
axes[1].set_title("Symmetric", fontsize=11)

fig.suptitle(f"Intervention on '{cols[i]}'", fontsize=12)

axes[1].legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=8,
    frameon=False,
    labelspacing=1,
    handlelength=1.5,
)

In [ ]:
select_idxes = [3, 7]

mean_asym_effect_ts = asym_effect_ts.mean(axis=(0, 1))
lo, hi = np.percentile(asym_effect_ts.mean(axis=0), q=(25, 75), axis=0)

fig, ax = plt.subplots(figsize=(5.5, 2), constrained_layout=True)

t = mean_asym_effect_ts.shape[0]
colours = ["tab:blue", "tab:green"]
for colour, col_idx in zip(colours, select_idxes, strict=True):
    ax.plot(
        np.arange(t),
        mean_asym_effect_ts[:, col_idx],
        label=f"{cols[col_idx]}",
        color=colour,
    )
    ax.fill_between(
        np.arange(t), lo[:, col_idx], hi[:, col_idx], alpha=0.3, color=colour
    )


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Time")
ax.set_ylabel("Effect of asymmetry")

# ax.set_ylim(-0.02, 0.02)
ax.set_xlim(0, 9)

ax.set_title(f"Intervention on '{cols[i]}'")


ax.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=8,
    frameon=False,
    labelspacing=1,
    handlelength=1.5,
);

## Illustrate difference in strategy between symmetric and asymmetric models

Look at possible interventions targeting 'Climate Policies'. 

### 1. How do the mean rankings of the collective effects of different interventions differ between symmetric and asymmetric models?

In [ ]:
from scipy.stats import rankdata


def bootstrap_mean_ranks_collective_effects(measurements, rng, n_boot=100, ci=95):
    # measurements: shape (n_individuals, repeats, n_interventions)
    M, R, N = measurements.shape
    boot_means = np.empty((n_boot, N))

    for b in range(n_boot):
        sample_rows = rng.integers(0, M, M)
        samples = measurements[sample_rows]

        # Collective effect as proportion of individuals with desired state
        collective_effect = ((samples + 1) // 2).mean(axis=0)

        # Rank interventions per-repeat
        ranks = rankdata(collective_effect, method="min", axis=-1)

        # Final estimate is the average rank per intervention spin
        boot_means[b] = ranks.mean(axis=0)

    mean = boot_means.mean(axis=0)
    ci = 1.97 * boot_means.std(axis=0, ddof=1) / np.sqrt(n_boot)
    lower = mean - ci
    upper = mean + ci
    # lower = np.percentile(boot_means, (100 - ci) / 2, axis=0)
    # upper = np.percentile(boot_means, 100 - (100 - ci) / 2, axis=0)

    return mean, lower, upper


i = 7

asym_mean, asym_lo, asym_hi = bootstrap_mean_ranks_collective_effects(
    int_asym_measurements[:, :, i], rng
)
sym_mean, sym_lo, sym_hi = bootstrap_mean_ranks_collective_effects(
    int_sym_measurements[:, :, i], rng
)

asym_mean = asym_mean[:-1]
asym_lo = asym_lo[:-1]
asym_hi = asym_hi[:-1]
sym_mean = sym_mean[:-1]
sym_lo = sym_lo[:-1]
sym_hi = sym_hi[:-1]

# asym_collective_effects = int_asym_measurements[:, :, i].sum(axis=0)
# sym_collective_effects = int_sym_measurements[:, :, i].sum(axis=0)
# asym_ranks = rankdata(mean_effect_by_ind_asym[:, i], method="min", axis=-1)
# asym_mean_ranks = asym_ranks.mean(axis=0)

# sym_ranks = rankdata(mean_effect_by_ind_sym[:, i], method="min", axis=-1)
# sym_mean_ranks = sym_ranks.mean(axis=0)

# asym_mean, asym_lo, asym_hi = bootstrap_mean_ranks(asym_ranks)
# sym_mean, sym_lo, sym_hi = bootstrap_mean_ranks(sym_ranks)

plot_df = pl.DataFrame(
    {
        "Model": ["Asymmetric"] * (n - 1) + ["Symmetric"] * (n - 1),
        "Intervention": np.concat((cols[:-1], cols[:-1])),
        "Mean rank": np.concat((asym_mean, sym_mean)),
        "CI low": np.concatenate([asym_lo, sym_lo]),
        "CI high": np.concatenate([asym_hi, sym_hi]),
    }
)

fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
sns.barplot(plot_df.to_pandas(), x="Intervention", y="Mean rank", hue="Model", ax=ax)

ax.set_xticks(np.arange(n - 1), cols[:-1], rotation=45, horizontalalignment="right")

x = np.arange(len(cols) - 1)
width = 0.4

for i, model in enumerate(["Asymmetric", "Symmetric"]):
    subset = plot_df.filter(Model=model)
    offset = -width / 2 if i == 0 else width / 2

    ax.errorbar(
        x + offset,
        subset["Mean rank"],
        yerr=[
            subset["Mean rank"] - subset["CI low"],
            subset["CI high"] - subset["Mean rank"],
        ],
        fmt="none",
        capsize=4,
        color="black",
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylim(1, 9)

leg = ax.get_legend()
handles = leg.legend_handles
labels = [t.get_text() for t in leg.texts]

# ax.legend(handles, labels)
leg.remove()

ax.legend(
    handles,
    labels,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    # fontsize=8,
    # handlelength=1,
    # columnspacing=0.5,
    # labelspacing=0.2,
    frameon=False,
)

### 2. How does mean effectiveness differ?

### 3. Or mean effect... (proportion of adoption)

In [ ]:
def bootstrap_mean_collective_effects(measurements, rng, n_boot=100, ci=95):
    # measurements: shape (n_individuals, repeats, n_interventions)
    M, R, N = measurements.shape
    boot_means = np.empty((n_boot, N))
    print(measurements[0, 0])

    for b in range(n_boot):
        sample_rows = rng.integers(0, M, M)
        samples = measurements[sample_rows]

        # Collective effect as proportion of individuals with desired state
        collective_effect = ((samples + 1) // 2).mean(axis=0)

        # Final estimate is the average rank per intervention spin
        boot_means[b] = collective_effect.mean(axis=0)

    mean = boot_means.mean(axis=0)
    ci = 1.97 * boot_means.std(axis=0, ddof=1) / np.sqrt(n_boot)
    lower = mean - ci
    upper = mean + ci
    # lower = np.percentile(boot_means, (100 - ci) / 2, axis=0)
    # upper = np.percentile(boot_means, 100 - (100 - ci) / 2, axis=0)

    return mean, lower, upper


i = 7

asym_mean, asym_lo, asym_hi = bootstrap_mean_collective_effects(
    int_asym_measurements[:, :, i], rng
)
sym_mean, sym_lo, sym_hi = bootstrap_mean_collective_effects(
    int_sym_measurements[:, :, i], rng
)

asym_mean = asym_mean[:-1]
asym_lo = asym_lo[:-1]
asym_hi = asym_hi[:-1]
sym_mean = sym_mean[:-1]
sym_lo = sym_lo[:-1]
sym_hi = sym_hi[:-1]

plot_df = pl.DataFrame(
    {
        "Model": ["Asymmetric"] * (n - 1) + ["Symmetric"] * (n - 1),
        "Intervention": np.concat((cols[:-1], cols[:-1])),
        "Proportion support": np.concat((asym_mean, sym_mean)),
        "CI low": np.concatenate([asym_lo, sym_lo]),
        "CI high": np.concatenate([asym_hi, sym_hi]),
    }
)

fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
sns.barplot(
    plot_df.to_pandas(), x="Intervention", y="Proportion support", hue="Model", ax=ax
)

ax.set_xticks(np.arange(n - 1), cols[:-1], rotation=45, horizontalalignment="right")

x = np.arange(len(cols) - 1)
width = 0.4

for j, model in enumerate(["Asymmetric", "Symmetric"]):
    subset = plot_df.filter(Model=model)
    offset = -width / 2 if j == 0 else width / 2

    ax.errorbar(
        x + offset,
        subset["Proportion support"],
        yerr=[
            subset["Proportion support"] - subset["CI low"],
            subset["CI high"] - subset["Proportion support"],
        ],
        fmt="none",
        capsize=4,
        color="black",
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylim(0.5, 0.65)

leg = ax.get_legend()
handles = leg.legend_handles
labels = [t.get_text() for t in leg.texts]

# ax.legend(handles, labels)
leg.remove()

ax.legend(
    handles,
    labels,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    # fontsize=8,
    # handlelength=1,
    # columnspacing=0.5,
    # labelspacing=0.2,
    frameon=False,
)

ax.axhline(
    y=((no_int_asym_measurements[:, :, i] + 1) // 2).mean(axis=(0, 1))[(i + 1) % n],
    linestyle="dashed",
    color="tab:blue",
    linewidth=1,
)
ax.axhline(
    y=((no_int_sym_measurements[:, :, i] + 1) // 2).mean(axis=(0, 1))[(i + 1) % n],
    linestyle="dashed",
    color="tab:orange",
    linewidth=1,
)

## RQ3.2

### 1. How do the mean rankings of different interventions differ between symmetric and asymmetric models for the average individual?

## Figure 1

> In the asymmetric model, the most effective intervention is typically one which targets an individual's political views.

For this figure, we want to show how targetting `Politics` compares to targetting other aspects of a belief system. We also want to show how this differs in the symmetric model.

One option here is to show all possible interventions for each target, as a bar chart.

Another option is to show only the top two (in terms of counterfactual shift), which shows how politics compares to the rest generally.

In [ ]:
data = int_effect_asym
mean_data = data.mean(axis=(0, 1))

In [ ]:
mean_data

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(mean_data[::-1], cmap=DIVERGING_CMAP, center=0.0, ax=ax)
ax.set_xticklabels(cols, rotation=90)
ax.set_yticklabels(cols[::-1], rotation=0)
ax.set_xlabel("Intervention node")
ax.set_ylabel("Outcome node")

In [ ]:
df = (
    pl.DataFrame(mean_data, schema=cols)
    .with_columns(pl.Series(cols).alias("outcome"))
    .unpivot(index="outcome", variable_name="intervene", value_name="effect")
    .filter(pl.col("effect") > 0)
)

In [ ]:
sns.barplot(df, x="outcome", y="effect", hue="intervene")

In [ ]:
sns.barplot(df, x="intervene", y="effect", hue="outcome")